# Data Generation for AI Doctor Patient Relationship

This notebook is ignored by Quarto because the filename starts with an underscore.

In [ ]:
import pandas as pd

# Use the 'resolve' URL for the raw file
url = "synthetic_patient_embeddings.parquet"

# Load the entire file
pats_df = pd.read_parquet(url, engine='pyarrow')
pats_df.head()

In [ ]:
# load the trial embeddings
url = "synthetic_trial_embeddings.parquet"

# Load the entire file
trial_df = pd.read_parquet(url, engine='pyarrow')
trial_df.head()

In [ ]:
# do a basic  keyword search to limit to NSCLC
pats_df['nsclc'] = pats_df.patient_summary.apply(lambda x: x.lower().count("lung cancer") > 1)
trial_df['nsclc'] = trial_df.this_space.apply(lambda x: x.lower().count("nsclc") > 1)

In [ ]:
pats_df[pats_df.nsclc].shape

In [ ]:
trial_df[trial_df.nsclc].shape

In [ ]:
print(pats_df.patient_summary.values[9])

In [ ]:
import numpy as np

# 1. Extract the patient embedding of interest (index 9)
target_patient_emb = np.array(pats_df.iloc[9]['patient_embedding'])

# 2. Extract all trial embeddings as a matrix
trial_embs = np.stack(trial_df['embedding'].values)

# 3. Calculate Cosine Similarity
# Cosine Similarity = (A · B) / (||A|| * ||B||)
dot_product = np.dot(trial_embs, target_patient_emb)
norms = np.linalg.norm(trial_embs, axis=1) * np.linalg.norm(target_patient_emb)
similarities = dot_product / norms

# 4. Assign back to trial_df and sort
trial_df['similarity'] = similarities
top_trials = trial_df.sort_values('similarity', ascending=False)

top_trials[['nct_id', 'title', 'similarity']].head(10)

In [ ]:
import plotly.express as px
from sklearn.manifold import MDS
from sklearn.metrics.pairwise import cosine_distances

# 1. Define specific trials to include and filter range
top_ids = ["NCT05978401", "NCT06685653", "NCT05652868"]
mask_ids = trial_df['nct_id'].isin(top_ids)
mask_range = trial_df['similarity'] >= 0.6

# 2. Combine specific trials with a random sample from the >=0.6 range
specific_trials = trial_df[mask_ids].copy()
remaining_pool = trial_df[mask_range & ~mask_ids]
random_sample = remaining_pool.sample(min(len(remaining_pool), 20))

sampled_trials = pd.concat([specific_trials, random_sample]).sort_values('similarity', ascending=False)

# 3. Prepare data for MDS
all_embeddings = np.vstack([
    target_patient_emb.reshape(1, -1), 
    np.stack(sampled_trials['embedding'].values)
])

# Calculate pairwise Cosine Distance (1 - Similarity)
dist_matrix = cosine_distances(all_embeddings)

# 4. Run MDS (Multidimensional Scaling)
# MDS attempts to place points such that 2D distances match original pairwise distances
mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42, normalized_stress='auto')
coords = mds.fit_transform(dist_matrix)

# 5. Create a DataFrame for Plotting
plot_df = pd.DataFrame(coords, columns=['X', 'Y'])
plot_df['type'] = ['Patient'] + ['Trial'] * len(sampled_trials)
plot_df['similarity'] = [1.0] + sampled_trials['similarity'].tolist()
plot_df['title'] = ['Patient Summary'] + sampled_trials['title'].tolist()
plot_df['nct_id'] = ['N/A'] + sampled_trials['nct_id'].tolist()
plot_df['summary'] = [pats_df.iloc[9]['patient_summary']] + sampled_trials['this_space'].tolist()

# 6. Plot with Plotly
fig = px.scatter(
    plot_df, 
    x='X', 
    y='Y', 
    color='similarity', 
    symbol='type',
    hover_data=['title', 'nct_id', 'summary'],
    title='MDS Plot: Visualizing Cosine Similarity Distance',
    color_continuous_scale='Greens', 
    range_color=[0.6, 1.0],
    width=900,
    height=700
)

fig.update_traces(marker=dict(size=14, line=dict(width=1, color='DarkSlateGrey')))
fig.show()

In [ ]:
# CSV, not parquet: the post only needs 33 rows, and a committed parquet broke
# rendering whenever the reader's pyarrow was older than the writer's Arrow.
plot_df.to_csv("example_embedding.csv", index=False)

In [ ]:
for idx, x in trial_df.sort_values('similarity', ascending=False).head(10).iterrows():
    print(x['title'])
    print("--", x['nct_id'])
    print()

In [ ]:
plot_df.summary.values[0]